# 03 — Synthetic Baseline Demand

Generate the canonical hour-of-week occupancy field used by downstream pricing simulations, and validate that CCC peak occupancy hits the 0.92 target.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
from phillyparking.spatial.segments import build_segments
from phillyparking.spatial.zones import assign_zones
from phillyparking.demand.synthetic_baseline import baseline_occupancy
segments = assign_zones(build_segments())
print('segments:', segments.shape)


## Build baseline occupancy panel

In [ ]:
panel = baseline_occupancy(segments)
print('panel:', panel.shape)
panel.head()


## Hour-of-week heatmap by zone

In [ ]:
if 'hour_of_week' in panel.columns:
    pivot = panel.pivot_table(index='zone', columns='hour_of_week', values='occupancy', aggfunc='mean')
elif {'dow', 'hour'}.issubset(panel.columns):
    panel['hour_of_week'] = panel['dow'] * 24 + panel['hour']
    pivot = panel.pivot_table(index='zone', columns='hour_of_week', values='occupancy', aggfunc='mean')
else:
    pivot = panel.pivot_table(index='zone', values='occupancy', aggfunc='mean').to_frame().T
fig, ax = plt.subplots(figsize=(12, 3))
im = ax.imshow(pivot.values, aspect='auto', cmap='magma')
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
ax.set_xlabel('hour of week')
ax.set_title('Baseline occupancy by zone')
plt.colorbar(im, ax=ax)
plt.show()


## Peak vs. segment characteristics

In [ ]:
peak = panel.groupby('segment_id')['occupancy'].max().rename('peak_occ')
df = segments.merge(peak, on='segment_id', how='left')
df.groupby('zone')['peak_occ'].agg(['mean', 'max', 'count'])


## Validation: CCC peak should approach 0.92

In [ ]:
ccc_peak = df.loc[df['zone'] == 'CCC', 'peak_occ'].mean()
print(f'CCC mean peak occupancy: {ccc_peak:.3f} (target ~0.92)')
assert 0.85 <= ccc_peak <= 0.99, 'CCC peak occupancy out of expected band'


## Next steps

- Move to `04_elasticity_bayesian.ipynb` to estimate price-elasticity priors.
